# H-mode global confinement $\tau_E$ scaling
Reproduces the basic engineering-variable results of Verdoolaege *et al.* 2021, Nucl. Fusion **61** 076006 (DB5.2.3) from the IMAS-migrated H-mode database.

In [1]:
import numpy as np

import _simdb_common as sc

db = sc.get_db()
sims = sc.query_dataset(db, "hmode")
print(f"{len(sims)} shots in hmode")

6394 shots in hmode


## Scaling variables

| Variable | IMAS path | Unit |
|---|---|---|
| $\tau_{E,th}$ (TAUTH) | `summary/global_quantities/tau_energy/value` | s |
| $I_p$ | `summary/global_quantities/ip/value` | A |
| $B_t$ | `summary/global_quantities/b0/value` | T |
| $\bar n_e$ | `summary/line_average/n_e/value` | m^-3 |
| $P_{l,th}$ | `summary/global_quantities/power_loss/value` | W |
| $R_{geo}$ | `summary/global_quantities/r0/value` | m |
| $V$ | `summary/global_quantities/volume/value` | m^3 |
| $a$ | `summary/boundary/minor_radius/value` | m |
| $M_{eff}$ | `summary/volume_average/meff_hydrogenic/value` | AMU |
| $\delta$ | `equilibrium/time_slice(0)/boundary/triangularity` | — |
| TOK | `summary/machine` | — |
| PHASE | `temporary/constant_string0d` (by `identifier.name`) | — |
| SELDB5 | `temporary/constant_integer0d` (by `identifier.name`) | — |

Derived (paper definitions): $\kappa_a = V/(2\pi R_{geo}\,\pi a^2)$, $\epsilon = a/R_{geo}$.

In [2]:
rows = []
for i, sim in enumerate(sims, start=1):
    if i % 100 == 0 or i == len(sims):
        print(f"\r  {i}/{len(sims)} shots ({sim.alias})".ljust(80), end="", flush=True)

    md = sim.meta_dict()
    machine = md.get("machine", "")
    seldb5 = sc.temp(md, "SELDB5", n=0)
    n = len(seldb5)
    if n == 0:
        continue
    tau    = sc.path(md, "global_quantities", "tau_energy", "value", n=n)
    tauth2 = sc.temp(md, "TAUTH2", n=n)  # fallback for tau when TAUTH is missing (see logbook section 8)
    tauth1 = sc.temp(md, "TAUTH1", n=n)  # second fallback
    ip    = sc.path(md, "global_quantities", "ip", "value", n=n)
    bt    = np.abs(sc.path(md, "global_quantities", "b0", "value", n=n))
    nel   = sc.path(md, "line_average", "n_e", "value", n=n)
    plth  = sc.path(md, "global_quantities", "power_loss", "value", n=n)
    rgeo  = sc.path(md, "boundary", "geometric_axis_r", "value", n=n)
    vol   = sc.path(md, "global_quantities", "volume", "value", n=n)
    amin  = sc.path(md, "boundary", "minor_radius", "value", n=n)
    meff  = sc.path(md, "volume_average", "meff_hydrogenic", "value", n=n)
    delta = sc.temp(md, "DELTA", n=n)
    phase_raw = sc.temp_str(md, "PHASE", n=n)
    evap  = sc.path_str(md, "wall", "evaporation", "value", n=n)  # TAUC93 input (ASDEX)
    dalfdv = sc.temp(md, "DALFDV", n=n)                            # TAUC93 input (PDX)
    dalfmp = sc.temp(md, "DALFMP", n=n)                            # TAUC93 input (PDX)
    limmat = sc.path(md, "limiter", "material", "index", n=n)      # AUG/AUG-W split (limiter material index; W=2)
    if limmat.ndim == 0:                                           # shot-constant scalar -> broadcast to timeslices
        limmat = np.full(n, float(limmat))
    divmat = sc.temp_str(md, "DIVMAT", n=n)                        # JET-C/JET-ILW split (divertor material; "W"=ILW)

    n_slices = min(len(a) for a in (seldb5, tau, tauth2, tauth1, ip, bt, nel, plth, rgeo, vol, amin, meff,
                                     delta, phase_raw, evap, dalfdv, dalfmp, limmat, divmat))
    if n_slices == 0:
        continue
    for j in range(n_slices):
        rows.append((machine, tau[j], tauth2[j], tauth1[j], ip[j], bt[j], nel[j], plth[j], rgeo[j], vol[j],
                     amin[j], meff[j], delta[j], phase_raw[j], seldb5[j], evap[j], dalfdv[j], dalfmp[j],
                     limmat[j], divmat[j]))
print()

dtype = [("tok", "U16"), ("tau", float), ("tauth2", float), ("tauth1", float), ("ip", float), ("bt", float),
         ("nel", float), ("plth", float), ("rgeo", float), ("vol", float), ("amin", float), ("meff", float),
         ("delta", float), ("phase", object), ("seldb5", float), ("evap", object), ("dalfdv", float), ("dalfmp", float),
         ("limmat", float), ("divmat", object)]
data = np.array(rows, dtype=dtype)

N      = len(data)
TAU    = data["tau"]
TAUTH2 = data["tauth2"]
TAUTH1 = data["tauth1"]
IP     = data["ip"]
BT     = data["bt"]
NEL    = data["nel"]
PLTH   = data["plth"]
RGEO   = data["rgeo"]
VOL    = data["vol"]
AMIN   = data["amin"]
MEFF   = data["meff"]
DELTA  = data["delta"]
TOK    = data["tok"]
PHASE  = data["phase"]
SELDB5 = data["seldb5"]
EVAP   = data["evap"]
DALFDV = data["dalfdv"]
DALFMP = data["dalfmp"]
LIMMAT = data["limmat"]
DIVMAT = data["divmat"]

print("Done.")

  6394/6394 shots (hmode/TUMAN3M/2000032308)                                   
Done.


In [3]:
# Units and derived variables (formula in paper)
# tau_s: TAUTH, falling back to TAUTH2 then TAUTH1 when TAUTH is missing/non-positive.
tau_ok    = np.isfinite(TAU) & (TAU > 0)
tauth2_ok = np.isfinite(TAUTH2) & (TAUTH2 > 0)
tauth1_ok = np.isfinite(TAUTH1) & (TAUTH1 > 0)

tau_s = np.where(tau_ok, TAU, np.where(tauth2_ok, TAUTH2, np.where(tauth1_ok, TAUTH1, np.nan)))
n_from_tauth2 = np.sum(~tau_ok & tauth2_ok)
n_from_tauth1 = np.sum(~tau_ok & ~tauth2_ok & tauth1_ok)
print(f"tau_s: {n_from_tauth2} timeslices filled from TAUTH2, {n_from_tauth1} from TAUTH1 (TAUTH missing/non-positive)")

# TAUC93: closed-divertor correction to tau_s for ASDEX and PDX (paper section 2.2.1).
# ELMy branch only, since `sel` below is restricted to ELMy H-modes.
EVAP_str = EVAP.astype(str)
is_asdex = TOK == "ASDEX"
bo = np.isin(EVAP_str, ["BOROA", "BOROB"]).astype(float)
ca = (EVAP_str == "CARBH").astype(float)
tauc93_asdex = 1.0 / (1.5 - 0.1 * bo - 0.15 * ca)

is_pdx = TOK == "PDX"
tauc93_pdx = 1.0 / (0.5 * (DALFDV / DALFMP)) ** 0.4

TAUC93 = np.ones(N)
TAUC93 = np.where(is_asdex, tauc93_asdex, TAUC93)
TAUC93 = np.where(is_pdx, tauc93_pdx, TAUC93)
tau_s = tau_s * TAUC93
print(f"TAUC93 applied: ASDEX mean={np.nanmean(TAUC93[is_asdex]):.3f}, PDX mean={np.nanmean(TAUC93[is_pdx]):.3f}")

ip_ma    = np.abs(IP) / 1e6                     # [MA]
Bt_T     = np.abs(BT)                           # [T]
ne_19    = NEL / 1e19                           # [10^19 m^-3]
Ploss_MW = PLTH / 1e6                           # [MW]
kappa_a  = VOL / (2.0 * np.pi * RGEO * np.pi * AMIN**2)   # paper kappa_a
eps      = AMIN / RGEO                          # inverse aspect ratio
one_delta = 1.0 + DELTA                         # 1 + delta

# Subset selection: STD5 standard set, ELMy H-mode
PHASE_str = PHASE.astype(str)
std5 = (SELDB5 == 1)
elmy = np.char.startswith(PHASE_str, "HG") | np.char.startswith(PHASE_str, "HS")

# Check if theres a difference?
print(f"Is std5 == elmy?: {np.array_equal(std5, elmy)}")

if not np.array_equal(std5,elmy):
    print(f"No, they differ in: {np.sum(std5 != elmy)} places")
    

regressors = [tau_s, ip_ma, Bt_T, ne_19, Ploss_MW, RGEO, kappa_a, eps, MEFF]
finite = np.all([np.isfinite(p) for p in regressors], axis=0)
positive = (tau_s > 0) & (Ploss_MW > 0) & (ip_ma > 0) & (Bt_T > 0) & (ne_19 > 0)
sel = std5 & elmy & finite & positive

print(f"Total pulses     : {N}")
print(f"SELDB5 == 1      : {np.sum(std5)}")
print(f"ELMy H (HG/HS)   : {np.sum(elmy)}")
print(f"std5 + ELMy + valid: {np.sum(sel)}")
print("\nper machine (ELMy, valid):")
for tok in np.unique(TOK[sel]):
    print(f"  {tok:10s}: {np.sum((TOK == tok) & sel)}")

tau_s: 49 timeslices filled from TAUTH2, 2 from TAUTH1 (TAUTH missing/non-positive)
TAUC93 applied: ASDEX mean=0.683, PDX mean=nan
Is std5 == elmy?: False
No, they differ in: 2560 places
Total pulses     : 14153
SELDB5 == 1      : 7568
ELMy H (HG/HS)   : 7520
std5 + ELMy + valid: 6167

per machine (ELMy, valid):
  ASDEX     : 431
  AUG       : 2147
  CMOD      : 45
  COMPASS   : 16
  D3D       : 388
  JET       : 2652
  JFT2M     : 70
  JT60U     : 100
  MAST      : 43
  NSTX      : 185
  PBXM      : 59
  START     : 8
  TCV       : 11
  TDEV      : 10
  TFTR      : 2


C:\Users\curranf\AppData\Local\Temp\ipykernel_9188\563906769.py:27: RuntimeWarning: Mean of empty slice
  print(f"TAUC93 applied: ASDEX mean={np.nanmean(TAUC93[is_asdex]):.3f}, PDX mean={np.nanmean(TAUC93[is_pdx]):.3f}")


## Table 2 — engineering-variable ranges (STD5 ELMy H)
Compare with paper Table 9 (DB5.2.3-STD5 ELMy H). 

In [4]:
cols = {
    "tau_E,th [s]": tau_s[sel],
    "Ip [MA]":      ip_ma[sel],
    "Bt [T]":       Bt_T[sel],
    "ne [1e19]":    ne_19[sel],
    "Pl,th [MW]":   Ploss_MW[sel],
    "Rgeo [m]":     RGEO[sel],
    "1+delta":      one_delta[sel],
    "kappa_a":      kappa_a[sel],
    "eps":          eps[sel],
    "Meff":         MEFF[sel],
}
print(f"{'variable':14s} {'min':>9s} {'max':>9s} {'mean':>9s} {'median':>9s} {'std':>9s} {'n':>6s}")
for name, x in cols.items():
    n_nan = np.sum(~np.isfinite(x))
    print(f"{name:14s} {np.nanmin(x):9.4g} {np.nanmax(x):9.4g} {np.nanmean(x):9.4g} {np.nanmedian(x):9.4g} "
          f"{np.nanstd(x):9.4g} {len(x)-n_nan:>6d}" + (f"  ({n_nan} NaN)" if n_nan else ""))

variable             min       max      mean    median       std      n
tau_E,th [s]    0.002236     1.321     0.178     0.128    0.1486   6167
Ip [MA]           0.1593     5.134     1.402     1.012    0.8058   6167
Bt [T]            0.2613     5.821     2.148     2.192    0.6669   6167
ne [1e19]          1.166      42.6     5.979     5.547     3.205   6167
Pl,th [MW]        0.1464     32.67     8.096      6.69     5.322   6167
Rgeo [m]          0.2804       3.4     2.172     1.679    0.7043   6167
1+delta                1     1.906      1.27     1.251    0.1384   6162  (5 NaN)
kappa_a           0.9308     2.389      1.54     1.563    0.1868   6167
eps               0.1548    0.7831     0.323    0.3149   0.08312   6167
Meff                   1      3.89     1.918         2    0.2904   6167


## Engineering scaling (OLS in log space)
Power law $\tau_{E,th} = \alpha_0\, I_p^{\alpha_I} B_t^{\alpha_B} \bar n_e^{\alpha_n} P_{l,th}^{\alpha_P} R_{geo}^{\alpha_R} \kappa_a^{\alpha_\kappa} \epsilon^{\alpha_\epsilon} M_{eff}^{\alpha_M}$ (paper eq. 2), fitted by ordinary least squares on $\ln$-transformed data.

In [5]:
# Design matrix in log space (Ip in MA, ne in 1e19, Pl,th in MW — absorbed into intercept)
y = np.log(tau_s[sel])
regressors = {
    "ln Ip":   np.log(ip_ma[sel]),
    "ln Bt":   np.log(Bt_T[sel]),
    "ln ne":   np.log(ne_19[sel]),
    "ln Plth": np.log(Ploss_MW[sel]),
    "ln Rgeo": np.log(RGEO[sel]),
    "ln kapa": np.log(kappa_a[sel]),
    "ln eps":  np.log(eps[sel]),
    "ln Meff": np.log(MEFF[sel]),
}

def paper_weights(tok_arr):
    w = np.ones(len(tok_arr))
    for tok in np.unique(tok_arr):
        m = tok_arr == tok
        w[m] = 1.0 / (2.0 + np.sqrt(m.sum() / 4.0))
    return w

X = np.column_stack([np.ones_like(y)] + list(regressors.values()))
w  = paper_weights(TOK[sel])
sw = np.sqrt(w)
coef, *_ = np.linalg.lstsq(X * sw[:,None], y * sw, rcond=None)

names = ["ln alpha0"] + list(regressors.keys())
# IPB98(y,2) reference exponents (Table 7): aI,aB,an,aP,aR,akappa,aeps,aM
ipb98 = {"ln Ip":0.93, "ln Bt":0.15, "ln ne":0.41, "ln Plth":-0.69,
         "ln Rgeo":1.97, "ln kapa":0.78, "ln eps":0.58, "ln Meff":0.19}
# Paper Table 9, "ELMy H" row (WLS, STD5 ELMy H, no ITER-like constraint, no delta)
paper_wls = {"ln Ip":0.984, "ln Bt":0.203, "ln ne":0.2530, "ln Plth":-0.6658,
             "ln Rgeo":1.698, "ln kapa":0.930, "ln eps":0.387, "ln Meff":0.175}

print(f"{'param':10s} {'this OLS':>10s} {'paper WLS (Tbl9)':>18s} {'IPB98(y,2)':>12s}")
print(f"{'alpha0':10s} {np.exp(coef[0]):>10.4g} {0.0576:>18.4g} {0.0562:>12.4g}")
for nm, c in zip(names[1:], coef[1:]):
    ref_paper = paper_wls.get(nm, np.nan)
    ref_ipb98 = ipb98.get(nm, np.nan)
    print(f"{nm:10s} {c:>10.3f} {ref_paper:>18.3f} {ref_ipb98:>12.3f}")

resid = y - X @ coef
rmse = np.sqrt(np.mean(resid**2))
r2 = 1.0 - np.sum(resid**2) / np.sum((y - y.mean())**2)
print(f"\nN = {sel.sum()}   RMSE(log) = {rmse:.3f}   R^2 = {r2:.3f}   (paper N = 6233, RMSE = 0.18, R^2 = 0.96)")

param        this OLS   paper WLS (Tbl9)   IPB98(y,2)
alpha0        0.06361             0.0576       0.0562
ln Ip           0.954              0.984        0.930
ln Bt           0.235              0.203        0.150
ln ne           0.247              0.253        0.410
ln Plth        -0.640             -0.666       -0.690
ln Rgeo         1.699              1.698        1.970
ln kapa         0.953              0.930        0.780
ln eps          0.473              0.387        0.580
ln Meff         0.116              0.175        0.190

N = 6167   RMSE(log) = 0.199   R^2 = 0.947   (paper N = 6233, RMSE = 0.18, R^2 = 0.96)


## Table 8 — per-machine OLS scalings (all H-modes in STD5)
OLS on log-transformed data. 

In [6]:
from numpy.linalg import lstsq

def ols_fit(y_log, X):
    """OLS in log space; returns (coef, std_err, resid)."""
    coef, _, _, _ = lstsq(X, y_log, rcond=None)
    n, p = X.shape
    resid = y_log - X @ coef
    # unbiased variance of residuals
    sigma2 = np.sum(resid**2) / max(n - p, 1)
    cov = sigma2 * np.linalg.pinv(X.T @ X)
    se = np.sqrt(np.diag(cov))
    return coef, se, resid

def metrics(resid, y_log, p_free):
    """MdAPE (%), RMSE (log), R^2 — all on log scale"""
    n = len(resid)
    mdape = np.median(np.abs(resid / y_log)) * 100
    rmse  = np.sqrt(np.sum(resid**2) / n)              # no df correction, matches paper
    ss_res = np.sum(resid**2)
    ss_tot = np.sum((y_log - y_log.mean())**2)
    r2    = 1.0 - ss_res / ss_tot
    return mdape, rmse, r2

# Paper Table 8 uses "all H-modes" in STD5 (not just ELMy).
all_h = (SELDB5 == 1)
pos_all = (tau_s > 0) & (Ploss_MW > 0) & (ip_ma > 0) & (Bt_T > 0) & (ne_19 > 0)
mask = all_h & pos_all & np.isfinite(tau_s)

# log-arrays (NaN-safe; we apply device mask before building X)
LOG = {
    "Ip":    np.log(np.where(ip_ma    > 0, ip_ma,    np.nan)),
    "Bt":    np.log(np.where(Bt_T     > 0, Bt_T,     np.nan)),
    "ne":    np.log(np.where(ne_19    > 0, ne_19,    np.nan)),
    "Plth":  np.log(np.where(Ploss_MW > 0, Ploss_MW, np.nan)),
    "Rgeo":  np.log(np.where(RGEO     > 0, RGEO,     np.nan)),
    "kappa": np.log(np.where(kappa_a  > 0, kappa_a,  np.nan)),
    "eps":   np.log(np.where(eps      > 0, eps,       np.nan)),
    "Meff":  np.log(np.where(MEFF     > 0, MEFF,     np.nan)),
    "1+d":   np.log(np.where(one_delta > 0, one_delta, np.nan)),
}
LOG_TAU = np.log(np.where(tau_s > 0, tau_s, np.nan))

# Wall-material splits (paper Table 1): AUG-W by limiter material index (W=2),
# JET-ILW by divertor material string ("W").
DIVMAT_str = DIVMAT.astype(str)
is_aug_w   = (TOK == "AUG") & (LIMMAT == 2)
is_jet_ilw = (TOK == "JET") & (DIVMAT_str == "W")

# (label, row_mask, [regressors]) -- order and predictor sets match paper Table 8
DEVICE_SPECS = [
    ("ASDEX",           TOK == "ASDEX",               ["Ip", "Bt", "ne", "Plth", "Meff"]),
    ("ASDEX Upgrade",   (TOK == "AUG") & ~is_aug_w,   ["Ip", "Bt", "ne", "Plth"]),
    ("ASDEX Upgrade W", is_aug_w,                     ["Ip", "Bt", "ne", "Plth"]),
    ("Alcator C-Mod",   TOK == "CMOD",                ["Ip",       "ne", "Plth"]),
    ("DIII-D",          TOK == "D3D",                 ["Ip", "Bt", "ne", "Plth", "1+d", "Meff"]),
    ("JET-C",           (TOK == "JET") & ~is_jet_ilw, ["Ip", "Bt", "ne", "Plth", "kappa", "Meff"]),
    ("JET-ILW",         is_jet_ilw,                   ["Ip", "Bt", "ne", "Plth", "Meff"]),
    ("JFT-2M",          TOK == "JFT2M",               ["Ip",       "ne", "Plth", "Meff"]),
    ("JT-60U",          TOK == "JT60U",               ["Ip", "Bt", "ne", "Plth"]),
    ("MAST",            TOK == "MAST",                ["Ip",       "ne", "Plth"]),
    ("NSTX",            TOK == "NSTX",                ["Ip", "Bt", "ne", "Plth", "kappa"]),
    ("PBX-M",           TOK == "PBXM",                ["Ip",       "ne", "Plth"]),
    ("PDX",             TOK == "PDX",                 ["Ip", "Bt", "ne", "Plth"]),
]

# paper Table 8 exponent columns (in order of predictors listed above)
COL_ORDER = ["Ip", "Bt", "ne", "Plth", "1+d", "kappa", "Meff"]
COL_LABELS = [r"$\alpha_I$", r"$\alpha_B$", r"$\alpha_n$", r"$\alpha_P$", 
              r"$\alpha_{(1+\delta)}$", r"$\kappa$", r"$\alpha_M$"]

header = (f"{'Device':<16s}  {'n_obs':>5s}  "
       + "   ".join(f"{c:>9s}" for c in COL_LABELS)
       + f"  {'MdAPE%':>7s}  {'RMSE':>6s}  {'R^2':>5s}")
print(header)
print("-" * len(header))

results = {}
for label, dev_mask, pkeys in DEVICE_SPECS:
    # require all regressors for this device to be finite
    log_preds = [LOG[k] for k in pkeys]
    finite_dev = np.all([np.isfinite(lp) for lp in log_preds], axis=0)
    finite_dev &= np.isfinite(LOG_TAU)
    m = mask & dev_mask & finite_dev
    n = int(m.sum())
    if n < len(pkeys) + 2:
        print(f"{label:<16s} {n:>5d}  (too few points)")
        continue

    y = LOG_TAU[m]
    cols_X = [np.ones(n)] + [LOG[k][m] for k in pkeys]
    X = np.column_stack(cols_X)
    coef, se, resid = ols_fit(y, X)
    mdape, rmse, r2 = metrics(resid, y, len(pkeys))

    # build coefficient dict keyed by predictor name
    coef_dict = {k: (coef[i+1], se[i+1]) for i, k in enumerate(pkeys)}

    # format row: show value pm standard error (se) for each column in COL_ORDER, blank if not fitted
    row = f"{label:<16s} {n:>5d}  "
    for col in COL_ORDER:
        if col in coef_dict:
            v, s = coef_dict[col]
            row += f"  {v:+5.3f}pm{s:5.3f}"
        else:
            row += f"  {'-------------':>9s}"
    row += f"  {mdape:7.1f}  {rmse:6.2f}  {r2:5.2f}"
    print(row)
    results[label] = {"n": n, "coef": coef_dict, "mdape": mdape, "rmse": rmse, "r2": r2}


Device            n_obs  $\alpha_I$   $\alpha_B$   $\alpha_n$   $\alpha_P$   $\alpha_{(1+\delta)}$    $\kappa$   $\alpha_M$   MdAPE%    RMSE    R^2
---------------------------------------------------------------------------------------------------------------------------------------------------
ASDEX              575    +0.751pm0.042  +0.205pm0.059  +0.664pm0.032  -0.634pm0.019  -------------  -------------  +0.262pm0.045      2.6    0.12   0.80
ASDEX Upgrade     1388    +1.618pm0.042  -0.260pm0.044  -0.070pm0.022  -0.685pm0.016  -------------  -------------  -------------      5.5    0.20   0.69
ASDEX Upgrade W    767    +1.512pm0.037  -0.356pm0.045  +0.069pm0.030  -0.540pm0.014  -------------  -------------  -------------      2.8    0.12   0.88
Alcator C-Mod       82    +1.149pm0.080  -------------  +0.101pm0.091  -0.597pm0.074  -------------  -------------  -------------      2.7    0.10   0.79
DIII-D             502    +1.101pm0.047  +0.100pm0.058  +0.094pm0.035  -0.633pm0.021  +0

All non-B_t scaling match table 8 coefficients, RMSE, $R^2$, and MdAPE *exactly*, except JFT-2M, which has an additional pulse, with coefficients consistent with an extra data point.

ASDEX: B, I, M coefficients differ significantly; n, P match exactly.